In [ ]:
!pip install -q transformers==5.8.0 accelerate bitsandbytes pillow

In [ ]:
"""
Chapter 11: Build a Vision Fine-Tuning Dataset (Colab Version)
===============================================================
Run this on Google Colab with an H100 runtime. It generates synthetic
UI screenshots, runs Gemma 4 31B locally as the teacher VLM, and
produces a JSONL dataset ready for vision fine-tuning.

No remote endpoints needed — everything runs on the Colab GPU.

Setup:
    1. Open Google Colab and select an H100 runtime
    2. Accept the Gemma 4 license at https://huggingface.co/google/gemma-4-31B-it
    3. Add your HF token to Colab secrets (key: HF_TOKEN)
    4. Run each cell in order
"""

# ── Cell 1: Install dependencies ─────────────────────────────
# !pip install -q transformers accelerate bitsandbytes pillow

# ── Cell 2: Configuration ────────────────────────────────────
import os
import json
import random
import base64
from pathlib import Path

from PIL import Image, ImageDraw, ImageFont

# Authenticate with Hugging Face
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

MODEL_ID = "google/gemma-4-31B-it"
OUTPUT_DIR = "./images/ui_screenshots"
DATASET_FILE = "./data/ui_vision_dataset.jsonl"
NUM_IMAGES = 250
TRAIN_SPLIT = 0.9

# ── Cell 3: Generate synthetic screenshots ───────────────────
# (Self-contained version of generate_synthetic_screenshots.py)

WIDTH, HEIGHT = 390, 844

PALETTES = [
    {"bg": "#FFFFFF", "surface": "#F5F5F5", "primary": "#1976D2",
     "primary_text": "#FFFFFF", "text": "#212121", "text_secondary": "#757575",
     "accent": "#FF5722", "divider": "#E0E0E0", "nav_bg": "#FAFAFA",
     "card_bg": "#FFFFFF", "input_bg": "#F5F5F5"},
    {"bg": "#121212", "surface": "#1E1E1E", "primary": "#BB86FC",
     "primary_text": "#000000", "text": "#E0E0E0", "text_secondary": "#A0A0A0",
     "accent": "#03DAC6", "divider": "#333333", "nav_bg": "#1E1E1E",
     "card_bg": "#2C2C2C", "input_bg": "#333333"},
    {"bg": "#FFF8F0", "surface": "#FFF0E0", "primary": "#E65100",
     "primary_text": "#FFFFFF", "text": "#3E2723", "text_secondary": "#8D6E63",
     "accent": "#FF6D00", "divider": "#FFE0B2", "nav_bg": "#FFF3E0",
     "card_bg": "#FFFFFF", "input_bg": "#FFF3E0"},
    {"bg": "#F0FFF4", "surface": "#E8F5E9", "primary": "#2E7D32",
     "primary_text": "#FFFFFF", "text": "#1B5E20", "text_secondary": "#66BB6A",
     "accent": "#00C853", "divider": "#C8E6C9", "nav_bg": "#E8F5E9",
     "card_bg": "#FFFFFF", "input_bg": "#F1F8E9"},
    {"bg": "#F3E5F5", "surface": "#EDE7F6", "primary": "#7B1FA2",
     "primary_text": "#FFFFFF", "text": "#311B92", "text_secondary": "#9575CD",
     "accent": "#E040FB", "divider": "#D1C4E9", "nav_bg": "#EDE7F6",
     "card_bg": "#FFFFFF", "input_bg": "#F3E5F5"},
    {"bg": "#FAFAFA", "surface": "#F0F0F0", "primary": "#424242",
     "primary_text": "#FFFFFF", "text": "#212121", "text_secondary": "#9E9E9E",
     "accent": "#FF4081", "divider": "#E0E0E0", "nav_bg": "#FFFFFF",
     "card_bg": "#FFFFFF", "input_bg": "#EEEEEE"},
]

SCREEN_TYPES = [
    "login", "signup", "home_feed", "settings", "profile",
    "product_detail", "search_results", "chat", "notifications",
    "onboarding", "checkout", "dashboard", "list_detail",
]

APP_NAMES = ["Taskly", "Greenleaf", "Pulse", "NovaPay", "CloudNine",
             "FitTrack", "MealPrep", "BookShelf", "TripWise", "StudyBuddy"]
HEADINGS = ["Welcome Back", "Create Account", "Your Dashboard", "Settings",
            "My Profile", "Product Details", "Search Results", "Messages",
            "Notifications", "Get Started", "Checkout", "Activity"]
BODY_TEXTS = [
    "Track your daily progress and reach your goals faster.",
    "Enter your email address to continue.",
    "Your order has been shipped and is on the way!",
    "No new notifications at this time.",
    "Manage your account settings and preferences.",
    "Discover new features and updates.",
    "Review your recent transactions below.",
    "Complete your profile to get personalized recommendations.",
    "Browse our latest collection of products.",
    "Enable two-factor authentication for added security.",
]
BUTTON_LABELS = ["Sign In", "Sign Up", "Continue", "Get Started", "Submit",
                 "Save Changes", "Add to Cart", "Buy Now", "Send", "Next",
                 "Confirm", "Apply", "Search", "Update", "Done"]
LIST_ITEMS = ["Account Settings", "Notifications", "Privacy & Security",
              "Payment Methods", "Help & Support", "About", "Language",
              "Dark Mode", "Log Out", "Edit Profile", "Change Password",
              "Linked Accounts", "Data & Storage", "Appearance"]
USERNAMES = ["alex_design", "sarah.codes", "mike_photo", "emily_writes",
             "jordan_dev", "casey_art", "taylor.run", "morgan_cook"]
CHAT_MESSAGES = ["Hey! Are you free for lunch?", "Just finished the project",
                 "Can you review my PR?", "Meeting moved to 3pm",
                 "Great work on the presentation!", "See you tomorrow",
                 "Did you see the latest update?", "Thanks for the help!"]
NAV_ICONS = ["Home", "Search", "Add", "Inbox", "Profile"]
PRODUCT_NAMES = ["Wireless Headphones", "Running Shoes", "Ceramic Mug",
                 "Laptop Stand", "Yoga Mat", "Water Bottle", "Desk Lamp",
                 "Backpack", "Smart Watch", "Plant Pot"]

QUESTION_TEMPLATES = [
    "Analyze this mobile app screenshot.",
    "Describe the UI layout and elements in this screenshot.",
    "What can you tell me about this app screen?",
    "Review this screenshot for layout and accessibility.",
    "Provide a detailed description of this user interface.",
    "What UI elements are visible and how are they arranged?",
    "Examine this screenshot and describe the user interface design.",
    "Break down the visual layout of this app screen.",
]

def hex_to_rgb(h):
    h = h.lstrip("#")
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))

def get_font(size, bold=False):
    paths = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf" if bold
        else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
    ]
    for p in paths:
        if os.path.exists(p):
            try:
                return ImageFont.truetype(p, size)
            except Exception:
                continue
    return ImageFont.load_default()


class ScreenGenerator:
    def __init__(self):
        self.w, self.h = WIDTH, HEIGHT
        self.palette = random.choice(PALETTES)
        self.app_name = random.choice(APP_NAMES)
        self.img = Image.new("RGB", (self.w, self.h),
                             hex_to_rgb(self.palette["bg"]))
        self.draw = ImageDraw.Draw(self.img)
        self.y = 0
        self.font_title = get_font(22, bold=True)
        self.font_heading = get_font(18, bold=True)
        self.font_body = get_font(14)
        self.font_small = get_font(12)
        self.font_button = get_font(15, bold=True)
        self.font_nav = get_font(10)

    def c(self, key):
        return hex_to_rgb(self.palette[key])

    def status_bar(self):
        self.draw.rectangle([0, 0, self.w, 44], fill=self.c("bg"))
        self.draw.text((20, 14), "9:41", fill=self.c("text"),
                       font=self.font_body)
        bx = self.w - 45
        self.draw.rectangle([bx, 16, bx+25, 28], outline=self.c("text"))
        self.draw.rectangle([bx+2, 18, bx+18, 26], fill=self.c("primary"))
        self.y = 44

    def nav_header(self, title, back=False, right=None):
        h = 48
        self.draw.rectangle([0, self.y, self.w, self.y+h],
                           fill=self.c("nav_bg"))
        self.draw.line([0, self.y+h-1, self.w, self.y+h-1],
                      fill=self.c("divider"))
        if back:
            self.draw.text((16, self.y+14), "<", fill=self.c("primary"),
                          font=self.font_heading)
        self.draw.text((50 if back else 20, self.y+14), title,
                      fill=self.c("text"), font=self.font_heading)
        if right:
            bbox = self.draw.textbbox((0,0), right, font=self.font_body)
            self.draw.text((self.w - (bbox[2]-bbox[0]) - 20, self.y+16),
                          right, fill=self.c("primary"), font=self.font_body)
        self.y += h

    def text_field(self, placeholder, filled=False):
        m, h = 24, 48
        self.draw.rounded_rectangle([m, self.y, self.w-m, self.y+h],
                                   radius=10, fill=self.c("input_bg"),
                                   outline=self.c("divider"))
        txt = random.choice(["alex@email.com", "johndoe", "********"]) \
            if filled else placeholder
        self.draw.text((m+16, self.y+15), txt,
                      fill=self.c("text" if filled else "text_secondary"),
                      font=self.font_body)
        self.y += h + 12

    def button(self, label, outlined=False):
        m, h = 24, 50
        if outlined:
            self.draw.rounded_rectangle([m, self.y, self.w-m, self.y+h],
                                       radius=12, outline=self.c("primary"),
                                       width=2)
            tc = self.c("primary")
        else:
            self.draw.rounded_rectangle([m, self.y, self.w-m, self.y+h],
                                       radius=12, fill=self.c("primary"))
            tc = self.c("primary_text")
        bbox = self.draw.textbbox((0,0), label, font=self.font_button)
        tw = bbox[2] - bbox[0]
        self.draw.text(((self.w - tw)//2, self.y+15), label, fill=tc,
                      font=self.font_button)
        self.y += h + 16

    def card(self, title, subtitle="", has_image=False):
        m = 20
        ch = 140 if has_image else 90
        self.draw.rounded_rectangle([m+2, self.y+2, self.w-m+2, self.y+ch+2],
                                   radius=12, fill=self.c("divider"))
        self.draw.rounded_rectangle([m, self.y, self.w-m, self.y+ch],
                                   radius=12, fill=self.c("card_bg"),
                                   outline=self.c("divider"))
        cy = self.y + 12
        if has_image:
            self.draw.rounded_rectangle([m+12, cy, m+80, cy+65], radius=8,
                                       fill=self.c("primary"))
            self.draw.text((m+96, cy+4), title, fill=self.c("text"),
                          font=self.font_body)
            if subtitle:
                self.draw.text((m+96, cy+24), subtitle,
                              fill=self.c("text_secondary"),
                              font=self.font_small)
        else:
            self.draw.text((m+16, cy+4), title, fill=self.c("text"),
                          font=self.font_body)
            if subtitle:
                self.draw.text((m+16, cy+26), subtitle,
                              fill=self.c("text_secondary"),
                              font=self.font_small)
        self.y += ch + 12

    def list_item(self, text, toggle=False, chevron=True):
        h = 52
        self.draw.line([24, self.y+h-1, self.w-24, self.y+h-1],
                      fill=self.c("divider"))
        self.draw.text((24, self.y+16), text, fill=self.c("text"),
                      font=self.font_body)
        if toggle:
            on = random.choice([True, False])
            tx = self.w - 70
            self.draw.rounded_rectangle(
                [tx, self.y+16, tx+44, self.y+36], radius=10,
                fill=self.c("primary") if on else self.c("divider"))
            cx = tx + 32 if on else tx + 12
            self.draw.ellipse([cx-8, self.y+18, cx+8, self.y+34],
                            fill=(255,255,255))
        elif chevron:
            self.draw.text((self.w-40, self.y+16), ">",
                          fill=self.c("text_secondary"), font=self.font_body)
        self.y += h

    def bottom_nav(self):
        by = self.h - 70
        self.draw.rectangle([0, by, self.w, self.h], fill=self.c("nav_bg"))
        self.draw.line([0, by, self.w, by], fill=self.c("divider"))
        active = random.randint(0, 4)
        for i, label in enumerate(NAV_ICONS):
            cx = (self.w // 5) * i + self.w // 10
            col = "primary" if i == active else "text_secondary"
            self.draw.ellipse([cx-10, by+10, cx+10, by+30], fill=self.c(col))
            bbox = self.draw.textbbox((0,0), label, font=self.font_nav)
            self.draw.text((cx - (bbox[2]-bbox[0])//2, by+36), label,
                          fill=self.c(col), font=self.font_nav)

    def avatar(self, x, y, size=40):
        colors = ["#E91E63", "#9C27B0", "#3F51B5", "#009688", "#FF9800"]
        self.draw.ellipse([x, y, x+size, y+size],
                         fill=hex_to_rgb(random.choice(colors)))
        ini = random.choice("ABCDEFGHIJKLM")
        bbox = self.draw.textbbox((0,0), ini, font=self.font_body)
        self.draw.text((x + (size-(bbox[2]-bbox[0]))//2,
                       y + (size-(bbox[3]-bbox[1]))//2 - 2),
                      ini, fill=(255,255,255), font=self.font_body)

    def search_bar(self):
        m, h = 20, 44
        self.draw.rounded_rectangle([m, self.y, self.w-m, self.y+h],
                                   radius=22, fill=self.c("input_bg"))
        self.draw.text((m+36, self.y+13), "Search...",
                      fill=self.c("text_secondary"), font=self.font_body)
        self.y += h + 12

    def chat_bubble(self, text, mine=False):
        max_w = self.w * 2 // 3
        words, lines, cur = text.split(), [], ""
        for w in words:
            test = f"{cur} {w}".strip()
            bbox = self.draw.textbbox((0,0), test, font=self.font_body)
            if bbox[2]-bbox[0] > max_w - 30:
                lines.append(cur); cur = w
            else:
                cur = test
        if cur: lines.append(cur)
        bh = len(lines) * 20 + 16
        if mine:
            x1 = self.w - 20 - max_w
            bg, tc = self.c("primary"), self.c("primary_text")
        else:
            x1 = 20
            bg, tc = self.c("surface"), self.c("text")
        self.draw.rounded_rectangle([x1, self.y, x1+max_w, self.y+bh],
                                   radius=16, fill=bg)
        for i, line in enumerate(lines):
            self.draw.text((x1+14, self.y+8+i*20), line, fill=tc,
                          font=self.font_body)
        self.y += bh + 8

    # ── Screen types ──────────────────────────────────────────
    def gen_login(self):
        self.status_bar(); self.y += 60
        cx = self.w // 2
        self.draw.rounded_rectangle([cx-30, self.y, cx+30, self.y+60],
                                   radius=15, fill=self.c("primary"))
        self.y += 80
        self.draw.text((cx-60, self.y), f"Welcome to {self.app_name}",
                      fill=self.c("text"), font=self.font_heading)
        self.y += 40
        self.text_field("Email address"); self.text_field("Password")
        self.y += 8; self.button("Sign In")
        self.button("Create Account", outlined=True)

    def gen_signup(self):
        self.status_bar(); self.nav_header("Create Account", back=True)
        self.y += 20
        self.draw.text((24, self.y), "Join " + self.app_name,
                      fill=self.c("text"), font=self.font_heading)
        self.y += 32
        for f in ["Full name", "Email address", "Password", "Confirm password"]:
            self.text_field(f)
        self.y += 8; self.button("Sign Up")

    def gen_home_feed(self):
        self.status_bar(); self.nav_header(self.app_name, right="Filter")
        self.search_bar()
        for _ in range(random.randint(3,5)):
            if self.y > self.h - 150: break
            self.card(random.choice(HEADINGS), random.choice(BODY_TEXTS),
                     has_image=random.random() > 0.3)
        self.bottom_nav()

    def gen_settings(self):
        self.status_bar(); self.nav_header("Settings", back=True)
        toggles = {"Dark Mode", "Notifications"}
        for item in random.sample(LIST_ITEMS, min(10, len(LIST_ITEMS))):
            if self.y > self.h - 140: break
            self.list_item(item, toggle=item in toggles,
                          chevron=item not in toggles)
        self.bottom_nav()

    def gen_profile(self):
        self.status_bar(); self.nav_header("Profile", right="Edit")
        self.y += 20; cx = self.w // 2
        self.avatar(cx-40, self.y, 80); self.y += 90
        name = random.choice(USERNAMES)
        bbox = self.draw.textbbox((0,0), name, font=self.font_heading)
        self.draw.text((cx-(bbox[2]-bbox[0])//2, self.y), name,
                      fill=self.c("text"), font=self.font_heading)
        self.y += 50
        stats = [("Posts", random.randint(10,999)),
                 ("Followers", random.randint(50,9999)),
                 ("Following", random.randint(50,500))]
        for i, (lab, val) in enumerate(stats):
            x = (self.w//3)*i + self.w//6
            self.draw.text((x-15, self.y), str(val), fill=self.c("text"),
                          font=self.font_heading)
            self.draw.text((x-15, self.y+24), lab,
                          fill=self.c("text_secondary"), font=self.font_small)
        self.y += 60; self.button("Follow"); self.bottom_nav()

    def gen_product_detail(self):
        self.status_bar(); self.nav_header("Details", back=True, right="Share")
        ih = 220
        self.draw.rectangle([0, self.y, self.w, self.y+ih],
                           fill=self.c("surface"))
        self.y += ih + 16
        self.draw.text((24, self.y), random.choice(PRODUCT_NAMES),
                      fill=self.c("text"), font=self.font_heading)
        self.y += 28
        self.draw.text((24, self.y),
                      f"${random.randint(9,299)}.{random.randint(0,99):02d}",
                      fill=self.c("primary"), font=self.font_heading)
        self.y += 32
        self.draw.text((24, self.y), random.choice(BODY_TEXTS),
                      fill=self.c("text_secondary"), font=self.font_body)
        self.y += 40; self.button("Add to Cart")
        self.button("Buy Now", outlined=True)

    def gen_search_results(self):
        self.status_bar(); self.nav_header("Search", back=True)
        self.search_bar()
        self.y += 4
        for _ in range(random.randint(3,6)):
            if self.y > self.h - 150: break
            self.card(random.choice(PRODUCT_NAMES),
                     f"${random.randint(9,299)}.{random.randint(0,99):02d}",
                     has_image=True)
        self.bottom_nav()

    def gen_chat(self):
        self.status_bar(); self.nav_header(random.choice(USERNAMES), back=True)
        self.y += 8
        for i, msg in enumerate(random.sample(CHAT_MESSAGES, min(6, len(CHAT_MESSAGES)))):
            if self.y > self.h - 160: break
            self.chat_bubble(msg, mine=(i % 2 == 1))
        by = self.h - 60
        self.draw.rectangle([0, by, self.w, self.h], fill=self.c("nav_bg"))
        self.draw.rounded_rectangle([16, by+10, self.w-70, by+44],
                                   radius=18, fill=self.c("input_bg"))
        self.draw.text((32, by+18), "Type a message...",
                      fill=self.c("text_secondary"), font=self.font_body)

    def gen_notifications(self):
        self.status_bar(); self.nav_header("Notifications", right="Clear")
        times = ["2m", "15m", "1h", "3h", "5h", "1d"]
        actions = ["liked your post", "started following you",
                   "commented on your photo", "sent you a message"]
        for _ in range(random.randint(4, 8)):
            if self.y > self.h - 150: break
            h = 72
            self.avatar(20, self.y+12, 40)
            self.draw.text((72, self.y+14), random.choice(USERNAMES),
                          fill=self.c("text"), font=self.font_body)
            self.draw.text((72, self.y+36), random.choice(actions),
                          fill=self.c("text_secondary"), font=self.font_small)
            self.draw.text((self.w-40, self.y+14), random.choice(times),
                          fill=self.c("text_secondary"), font=self.font_small)
            self.draw.line([72, self.y+h-1, self.w-20, self.y+h-1],
                          fill=self.c("divider"))
            self.y += h
        self.bottom_nav()

    def gen_onboarding(self):
        self.status_bar(); self.y += 80
        cx = self.w // 2
        self.draw.rounded_rectangle([cx-80, self.y, cx+80, self.y+160],
                                   radius=20, fill=self.c("surface"))
        self.y += 190
        heading = random.choice(HEADINGS)
        bbox = self.draw.textbbox((0,0), heading, font=self.font_title)
        self.draw.text((cx-(bbox[2]-bbox[0])//2, self.y), heading,
                      fill=self.c("text"), font=self.font_title)
        self.y += 36
        body = random.choice(BODY_TEXTS)[:45]
        bbox = self.draw.textbbox((0,0), body, font=self.font_body)
        self.draw.text((cx-(bbox[2]-bbox[0])//2, self.y), body,
                      fill=self.c("text_secondary"), font=self.font_body)
        self.y += 60
        self.button(random.choice(["Get Started", "Next", "Continue"]))
        self.button("Skip", outlined=True)

    def gen_checkout(self):
        self.status_bar(); self.nav_header("Checkout", back=True)
        self.y += 8
        self.draw.text((24, self.y), "Order Summary",
                      fill=self.c("text"), font=self.font_heading)
        self.y += 30
        total = 0
        for _ in range(random.randint(2,4)):
            p = random.randint(9,99); total += p
            self.draw.text((24, self.y), random.choice(PRODUCT_NAMES),
                          fill=self.c("text"), font=self.font_body)
            self.draw.text((self.w-70, self.y), f"${p}.00",
                          fill=self.c("text"), font=self.font_body)
            self.y += 28
        self.draw.line([24, self.y, self.w-24, self.y], fill=self.c("divider"))
        self.y += 12
        self.draw.text((24, self.y), "Total", fill=self.c("text"),
                      font=self.font_heading)
        self.draw.text((self.w-90, self.y), f"${total}.00",
                      fill=self.c("primary"), font=self.font_heading)
        self.y += 40; self.button(f"Pay ${total}.00")

    def gen_dashboard(self):
        self.status_bar(); self.nav_header(self.app_name, right="Settings")
        self.y += 8
        self.draw.text((24, self.y), random.choice(["Good morning!",
                       "Your Dashboard", "Weekly Summary"]),
                      fill=self.c("text"), font=self.font_heading)
        self.y += 36
        cw = (self.w - 60) // 2
        for row in range(2):
            for col in range(2):
                x = 20 + col * (cw + 16)
                self.draw.rounded_rectangle([x, self.y, x+cw, self.y+80],
                    radius=12, fill=self.c("card_bg"),
                    outline=self.c("divider"))
                val = random.choice(["$1,234", "89%", "156", "2.4K"])
                lab = random.choice(["Revenue", "Tasks Done", "Users", "Score"])
                self.draw.text((x+16, self.y+16), val,
                              fill=self.c("primary"), font=self.font_heading)
                self.draw.text((x+16, self.y+44), lab,
                              fill=self.c("text_secondary"),
                              font=self.font_small)
            self.y += 96
        self.bottom_nav()

    def gen_list_detail(self):
        self.status_bar()
        self.nav_header(random.choice(HEADINGS), back=True, right="Edit")
        self.y += 12
        self.draw.text((24, self.y), random.choice(BODY_TEXTS),
                      fill=self.c("text_secondary"), font=self.font_body)
        self.y += 32
        for item in random.sample(LIST_ITEMS, min(8, len(LIST_ITEMS))):
            if self.y > self.h - 120: break
            self.list_item(item)
        self.bottom_nav()

    def generate(self, screen_type=None):
        if screen_type is None:
            screen_type = random.choice(SCREEN_TYPES)
        gen_fn = {
            "login": self.gen_login, "signup": self.gen_signup,
            "home_feed": self.gen_home_feed, "settings": self.gen_settings,
            "profile": self.gen_profile,
            "product_detail": self.gen_product_detail,
            "search_results": self.gen_search_results,
            "chat": self.gen_chat, "notifications": self.gen_notifications,
            "onboarding": self.gen_onboarding, "checkout": self.gen_checkout,
            "dashboard": self.gen_dashboard,
            "list_detail": self.gen_list_detail,
        }.get(screen_type, self.gen_home_feed)
        gen_fn()
        return self.img


def generate_screenshots():
    """Generate synthetic UI screenshots."""
    out = Path(OUTPUT_DIR)
    out.mkdir(parents=True, exist_ok=True)
    print(f"Generating {NUM_IMAGES} synthetic UI screenshots...")

    paths = []
    for i in range(NUM_IMAGES):
        stype = SCREEN_TYPES[i % len(SCREEN_TYPES)]
        gen = ScreenGenerator()
        img = gen.generate(stype)
        fname = f"ui_{stype}_{i:03d}.png"
        fpath = out / fname
        img.save(fpath)
        paths.append(str(fpath))
        print(f"  [{i+1}/{NUM_IMAGES}] {fname}")

    print(f"Done! {NUM_IMAGES} screenshots saved to {out}/\n")
    return paths


# ── Cell 4: Generate the screenshots ─────────────────────────
image_paths = generate_screenshots()


# ── Cell 5: Load Gemma 4 31B ─────────────────────────────────
import torch
from transformers import AutoProcessor, Gemma4ForConditionalGeneration

print("Loading Gemma 4 31B (this takes a few minutes)...")
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Gemma4ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.eval()
print("Model loaded!\n")


# ── Cell 6: Define the teacher analysis function ─────────────
SYSTEM_PROMPT = """You are a senior UX analyst examining a mobile app screenshot.
Provide a structured analysis with these sections:

1. LAYOUT OVERVIEW: Describe the overall screen layout (navigation bars,
   content areas, floating elements).
2. UI ELEMENTS: List the key interactive elements (buttons, text fields,
   toggles, cards) with their approximate positions.
3. TEXT CONTENT: Transcribe any visible text on the screen.
4. ACCESSIBILITY NOTES: Flag potential accessibility issues (small tap
   targets, low contrast text, missing labels).
5. SUMMARY: One sentence describing the screen's purpose.

Be specific and factual. Describe what you see, not what you assume."""


def analyze_image(image_path: str) -> str | None:
    """Run Gemma 4 31B locally on a single image."""
    try:
        image = Image.open(image_path).convert("RGB")

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text",
                     "text": "Analyze this mobile app screenshot."},
                ],
            },
        ]

        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = processor(
            text=text, images=[image], return_tensors="pt"
        ).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs, max_new_tokens=1000, temperature=0.3, do_sample=True
            )

        generated = output_ids[0][inputs["input_ids"].shape[1]:]
        return processor.decode(generated, skip_special_tokens=True).strip()
    except Exception as e:
        print(f"  Error: {e}")
        return None


# ── Cell 7: Build the dataset ────────────────────────────────
print(f"Processing {len(image_paths)} images with Gemma 4 31B...\n")

examples = []
for i, img_path in enumerate(image_paths):
    fname = Path(img_path).name
    print(f"[{i+1}/{len(image_paths)}] {fname}...", end=" ", flush=True)

    description = analyze_image(img_path)
    if description is None:
        print("FAILED")
        continue

    print(f"OK ({len(description)} chars)")
    question = random.choice(QUESTION_TEMPLATES)
    examples.append({
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image", "url": img_path},
                    {"type": "text", "text": question},
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": description},
                ],
            },
        ]
    })

print(f"\nGenerated {len(examples)} examples from {len(image_paths)} images.")


# ── Cell 8: Save the dataset ─────────────────────────────────
random.shuffle(examples)
split_idx = int(len(examples) * TRAIN_SPLIT)
train_examples = examples[:split_idx]
val_examples = examples[split_idx:]

os.makedirs(os.path.dirname(DATASET_FILE), exist_ok=True)

with open(DATASET_FILE, "w") as f:
    for ex in examples:
        f.write(json.dumps(ex) + "\n")

train_file = DATASET_FILE.replace(".jsonl", "_train.jsonl")
with open(train_file, "w") as f:
    for ex in train_examples:
        f.write(json.dumps(ex) + "\n")

val_file = DATASET_FILE.replace(".jsonl", "_val.jsonl")
with open(val_file, "w") as f:
    for ex in val_examples:
        f.write(json.dumps(ex) + "\n")

print(f"\nDataset saved!")
print(f"  Total: {len(examples)} → {DATASET_FILE}")
print(f"  Train: {len(train_examples)} → {train_file}")
print(f"  Val:   {len(val_examples)} → {val_file}")


# ── Cell 9: Download the dataset ─────────────────────────────
# Uncomment these lines in Colab to download the files:
# from google.colab import files
# files.download(DATASET_FILE)
# files.download(train_file)
# files.download(val_file)


Generating 250 synthetic UI screenshots...
  [1/250] ui_login_000.png
  [2/250] ui_signup_001.png
  [3/250] ui_home_feed_002.png
  [4/250] ui_settings_003.png
  [5/250] ui_profile_004.png
  [6/250] ui_product_detail_005.png
  [7/250] ui_search_results_006.png
  [8/250] ui_chat_007.png
  [9/250] ui_notifications_008.png
  [10/250] ui_onboarding_009.png
  [11/250] ui_checkout_010.png
  [12/250] ui_dashboard_011.png
  [13/250] ui_list_detail_012.png
  [14/250] ui_login_013.png
  [15/250] ui_signup_014.png
  [16/250] ui_home_feed_015.png
  [17/250] ui_settings_016.png
  [18/250] ui_profile_017.png
  [19/250] ui_product_detail_018.png
  [20/250] ui_search_results_019.png
  [21/250] ui_chat_020.png
  [22/250] ui_notifications_021.png
  [23/250] ui_onboarding_022.png
  [24/250] ui_checkout_023.png
  [25/250] ui_dashboard_024.png
  [26/250] ui_list_detail_025.png
  [27/250] ui_login_026.png
  [28/250] ui_signup_027.png
  [29/250] ui_home_feed_028.png
  [30/250] ui_settings_029.png
  [31/250] u

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Model loaded!

Processing 250 images with Gemma 4 31B...

[1/250] ui_login_000.png... OK (1866 chars)
[2/250] ui_signup_001.png... OK (1650 chars)
[3/250] ui_home_feed_002.png... OK (2258 chars)
[4/250] ui_settings_003.png... OK (1937 chars)
[5/250] ui_profile_004.png... OK (2021 chars)
[6/250] ui_product_detail_005.png... OK (1825 chars)
[7/250] ui_search_results_006.png... OK (2069 chars)
[8/250] ui_chat_007.png... OK (1633 chars)
[9/250] ui_notifications_008.png... OK (2531 chars)
[10/250] ui_onboarding_009.png... OK (1490 chars)
[11/250] ui_checkout_010.png... OK (1667 chars)
[12/250] ui_dashboard_011.png... OK (1990 chars)
[13/250] ui_list_detail_012.png... OK (2256 chars)
[14/250] ui_login_013.png... OK (1887 chars)
[15/250] ui_signup_014.png... OK (1760 chars)
[16/250] ui_home_feed_015.png... OK (2498 chars)
[17/250] ui_settings_016.png... OK (2170 chars)
[18/250] ui_profile_017.png... OK (2023 chars)
[19/250] ui_product_detail_018.png... OK (1890 chars)
[20/250] ui_search_resul

In [ ]:
# ── Cell: Zip and download results ────────────────────────────
import zipfile
from pathlib import Path
from google.colab import files

zip_path = "vision_dataset_results.zip"
dirs_to_zip = ["./images", "./data", "./sample_data"]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for d in dirs_to_zip:
        p = Path(d)
        if not p.exists():
            print(f"Skipping {d} (not found)")
            continue
        for f in sorted(p.rglob("*")):
            if f.is_file():
                zf.write(f)
                print(f"  Added {f}")

print(f"\nZip created: {zip_path}")
files.download(zip_path)


  Added images/ui_screenshots/ui_chat_007.png
  Added images/ui_screenshots/ui_chat_020.png
  Added images/ui_screenshots/ui_chat_033.png
  Added images/ui_screenshots/ui_chat_046.png
  Added images/ui_screenshots/ui_chat_059.png
  Added images/ui_screenshots/ui_chat_072.png
  Added images/ui_screenshots/ui_chat_085.png
  Added images/ui_screenshots/ui_chat_098.png
  Added images/ui_screenshots/ui_chat_111.png
  Added images/ui_screenshots/ui_chat_124.png
  Added images/ui_screenshots/ui_chat_137.png
  Added images/ui_screenshots/ui_chat_150.png
  Added images/ui_screenshots/ui_chat_163.png
  Added images/ui_screenshots/ui_chat_176.png
  Added images/ui_screenshots/ui_chat_189.png
  Added images/ui_screenshots/ui_chat_202.png
  Added images/ui_screenshots/ui_chat_215.png
  Added images/ui_screenshots/ui_chat_228.png
  Added images/ui_screenshots/ui_chat_241.png
  Added images/ui_screenshots/ui_checkout_010.png
  Added images/ui_screenshots/ui_checkout_023.png
  Added images/ui_screensh

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>